In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# তোমার সঠিক পাথ
path = '/content/drive/MyDrive/Thesis file/Feature_Extract_Formants/'

extractors = ['AlexNet', 'CustomCNN', 'DenseNet121', 'EfficientNetB0', 'InceptionV3',
              'MobileNetV2', 'ResNet50', 'VGG16', 'ViT', 'Xception']

classifiers = {
    'AdaBoost': AdaBoostClassifier(n_estimators=100, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Extra Trees': ExtraTreesClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Logistic Regression': LogisticRegression(max_iter=3000, random_state=42),
    'Naive Bayes': GaussianNB(),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(random_state=42)
}

results = pd.DataFrame(columns=['Feature Extractor'] + list(classifiers.keys()) + ['Best', 'Best Accuracy'])
results['Feature Extractor'] = extractors

print("F1 + F4 কম্বিনেশন ক্লাসিফিকেশন শুরু হচ্ছে...\n")

for extractor in extractors:
    print(f'Processing {extractor} (F1 + F4) ...')

    f1_file = f'{path}F1_Features_{extractor}.csv'
    f4_file = f'{path}F4_Features_{extractor}.csv'   # ← এখানে F4

    df1 = pd.read_csv(f1_file)
    df4 = pd.read_csv(f4_file)                      # ← df4

    # লেবেল কলাম অটো ডিটেক্ট
    possible_label_cols = ['label', 'class', 'target', 'phoneme', 'category', 'Label', 'Class']
    label_col = None
    for col in possible_label_cols:
        if col in df1.columns:
            label_col = col
            break

    if label_col is None:
        label_col = df1.columns[1]
        print(f"   → লেবেল কলাম ধরা হলো: {label_col}")

    # ক্লিন করি
    def clean_features(df):
        df = df.copy()
        drop_cols = [c for c in df.columns if any(x in c.lower() for x in ['filename', 'path', 'unnamed', 'index', 'id'])]
        df.drop(columns=drop_cols, errors='ignore', inplace=True)
        return df

    df1_clean = clean_features(df1)
    df4_clean = clean_features(df4)                  # ← df4_clean

    X1 = df1_clean.drop(columns=[label_col], errors='ignore')
    X4 = df4_clean.drop(columns=[label_col], errors='ignore')  # ← X4
    y = df1_clean[label_col]

    # ফিচার জোড়া
    X = pd.concat([X1.reset_index(drop=True), X4.reset_index(drop=True)], axis=1)

    # যদি শুধু ১টা ক্লাস থাকে → স্কিপ (এরর আসবে না)
    if y.nunique() < 2:
        print(f"   → শুধু ১টা ক্লাস আছে → স্কিপ করা হলো")
        for clf_name in classifiers.keys():
            results.loc[results['Feature Extractor'] == extractor, clf_name] = "Only 1 class"
        results.loc[results['Feature Extractor'] == extractor, 'Best'] = "N/A"
        results.loc[results['Feature Extractor'] == extractor, 'Best Accuracy'] = "N/A"
        continue

    # স্প্লিট
    try:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    except:
        print(f"   → {extractor}: stratify ছাড়া স্প্লিট")
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    best_acc = 0
    best_name = ''

    for name, clf in classifiers.items():
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        acc = round(accuracy_score(y_test, y_pred), 4)
        results.loc[results['Feature Extractor'] == extractor, name] = acc
        if acc > best_acc:
            best_acc = acc
            best_name = name

    results.loc[results['Feature Extractor'] == extractor, 'Best'] = best_name
    results.loc[results['Feature Extractor'] == extractor, 'Best Accuracy'] = best_acc

# সেভ করি — নামে F1_F4 বোঝানো হলো
output_file = path + 'Full_Summary_Table_F1_F4.xlsx'
results.to_excel(output_file, index=False)

print('\nসব শেষ! F1 + F4 এর এক্সেল তৈরি হয়ে গেছে')
print(f"লোকেশন: {output_file}")
display(results)

Mounted at /content/drive
F1 + F4 কম্বিনেশন ক্লাসিফিকেশন শুরু হচ্ছে...

Processing AlexNet (F1 + F4) ...
Processing CustomCNN (F1 + F4) ...
